In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import tempfile
from pathlib import Path

if "tensorflow" in sys.modules and os.environ.get("TF_USE_LEGACY_KERAS") != "1":
    raise RuntimeError(
        "Restart the Colab session, then run this notebook from its first cell. "
        "NIF requires the legacy Keras setting before TensorFlow is imported."
    )
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

COLAB_REQUIREMENTS = [
    "tensorflow==2.21.0",
    "tf-keras==2.21.0",
    "keras==3.13.2",
    "tensorflow-probability==0.25.0",
    "tensorflow-model-optimization==0.8.1",
    "numpy==2.1.3",
    "scipy==1.16.3",
    "matplotlib==3.10.0",
    "tensorboard==2.21.0",
    "absl-py==1.4.0",
]
NIF_COMMIT = "ad9113df14590189b12068b6b3fa7c8c8d9187d4"

if sys.version_info < (3, 11) or sys.version_info >= (3, 14):
    raise RuntimeError(
        "Use a Colab Python 3.11–3.13 runtime for this pinned environment."
    )

missing = []
for requirement in COLAB_REQUIREMENTS:
    package, expected = requirement.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(requirement)
if missing:
    package_modules = {
        "tensorflow": "tensorflow",
        "tf-keras": "tf_keras",
        "keras": "keras",
        "numpy": "numpy",
        "scipy": "scipy",
        "matplotlib": "matplotlib",
        "tensorflow-probability": "tensorflow_probability",
        "tensorflow-model-optimization": "tensorflow_model_optimization",
        "tensorboard": "tensorboard",
        "absl-py": "absl",
    }
    changing = {item.split("==")[0] for item in missing}
    loaded = [
        module
        for package, module in package_modules.items()
        if package in changing and module in sys.modules
    ]
    if loaded:
        raise RuntimeError(
            "Restart the Colab session before changing packages; already imported: "
            + ", ".join(loaded)
        )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *COLAB_REQUIREMENTS,
        ]
    )

runtime_root = Path(tempfile.gettempdir()) / ("nif-tutorials-v2-" + NIF_COMMIT[:12])
stamp = runtime_root / ".nif-source-commit"
if not stamp.exists() or stamp.read_text().strip() != NIF_COMMIT:
    runtime_root.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "--upgrade",
            "--disable-pip-version-check",
            "--target",
            str(runtime_root),
            "git+https://github.com/pswpswpsw/nif.git@" + NIF_COMMIT,
        ]
    )
    stamp.write_text(NIF_COMMIT)


def replace_compatibility(path, old, new):
    content = path.read_text(encoding="utf-8")
    if new in content:
        return
    if old in content:
        path.write_text(content.replace(old, new), encoding="utf-8")
    else:
        raise RuntimeError("Pinned NIF source changed unexpectedly: " + str(path))


optimizer_file = runtime_root / "nif" / "optimizers" / "external_optimizers.py"
replace_compatibility(
    optimizer_file,
    "from tensorflow.python.distribute import distribution_strategy_context as distribute_ctx",
    "from tensorflow.python.distribute import distribute_lib as distribute_ctx",
)
for optimizer_class in ("AdaBeliefOptimizer", "L4Adam"):
    replace_compatibility(
        optimizer_file,
        "class " + optimizer_class + "(tf.keras.optimizers.Optimizer):",
        "class " + optimizer_class + "(tf.keras.optimizers.legacy.Optimizer):",
    )
for name in ("lbfgs.py", "lbfgs_V2.py"):
    path = runtime_root / "nif" / "optimizers" / name
    content = path.read_text(encoding="utf-8")
    if "np.product(" in content:
        path.write_text(content.replace("np.product(", "np.prod("), encoding="utf-8")

tfr_file = runtime_root / "nif" / "data" / "tfr_dataset.py"
replace_compatibility(
    tfr_file,
    "target = tf.transpose(\n"
    "                batch_file[self.n_feature : self.n_feature + self.n_target]\n            )",
    "target = tf.transpose(\n"
    "                tf.squeeze(tf.stack("
    "batch_file[self.n_feature : self.n_feature + self.n_target], 0), 1)\n"
    "            )",
)
replace_compatibility(
    tfr_file,
    "target = tf.transpose(batch_file[-self.n_target :])",
    "target = tf.transpose(\n"
    "                tf.squeeze(tf.stack(batch_file[-self.n_target :], 0), 1)\n            )",
)

gc_file = runtime_root / "nif" / "optimizers" / "gtcf.py"
replace_compatibility(
    gc_file,
    "    def get_centralized_gradients_for_optimizer(loss, params):\n",
    """    if not getattr(optimizer, "_nif_gradient_centralization", False):
        original_get_gradients = optimizer._get_gradients

        def centralized_tape_gradients(*args, **kwargs):
            gradients_and_variables = original_get_gradients(*args, **kwargs)
            return [
                (gradient - tf.reduce_mean(
                    gradient, axis=list(range(len(gradient.shape) - 1)), keepdims=True
                )
                if gradient is not None and len(gradient.shape) > 1 else gradient, variable)
                for gradient, variable in gradients_and_variables
            ]

        optimizer._get_gradients = centralized_tape_gradients
        optimizer._nif_gradient_centralization = True

    def get_centralized_gradients_for_optimizer(loss, params):
""",
)

if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
if "nif" in sys.modules:
    loaded_nif = Path(sys.modules["nif"].__file__).resolve()
    if not loaded_nif.is_relative_to(runtime_root.resolve()):
        raise RuntimeError(
            "Restart the session: an unrelated NIF installation is already imported."
        )
print("Python:", sys.version.split()[0])
for requirement in COLAB_REQUIREMENTS:
    package = requirement.split("==")[0]
    print(package + ":", importlib.metadata.version(package))
print("NIF source commit:", NIF_COMMIT)
print("Compatibility copy:", runtime_root / "nif")


# How to training on large dataset (>5GB) where data cannot fit in memory?

### Note 

- GPU memory is precious, don't waste it on dataset
- It can be super interesting to train NIF on HUGE dataset!
- In this Google Colab, I will show a demonstration using [tensorflow record](https://www.tensorflow.org/tutorials/load_data/tfrecord),
which is a simple format for storing a sequence of binary records.
- First you will learn how to obtain `tfrecord` file from `npz`, then you will
learn how to perform training across different `tfrecord` files.

## 0. Prepare a "large" numpy `npz` to play with

- this "ficitious" dataset has 7 dimensions, 
$$(t,x,y,z,u,v,w)$$
where $u,v,w$ are three components of velocity field. 
- Thus, we have **4 features, and 3 targets, no weight**. We will need this information to create the dataset 

- let's call it `Big_npz_file.npz`, it has 54 MB with $10^6$ points, each point contains 7 real number values
- it has a key argument `data`, which stores the data


In [ ]:
import numpy as np

data = np.random.uniform(0,1,(1000000,7))
np.savez('Big_npz_file.npz', data=data)

## 1. Generate `tfrecord` files from a big a `npz` 

First, you need to bring the `TFRDataset` class under `nif.data.tfr_dataset`

In [ ]:
from nif.data.tfr_dataset import TFRDataset

Second, create an instance, a "file handler", call it `fh`, given the `n_features = 3` and `n_target = 3`

In [ ]:
fh = TFRDataset(n_feature=4, n_target=3)

### Then, you need to decide 
1. How many points to store in one single **tfrecord** file ----> `num_pts_per_file`
    - it doesn't have to be perfectly dividing the total number of points, we will make the last one smaller if it doesn't have a perfect divide.
    - Tensorflow official document [suggest](https://www.tensorflow.org/tutorials/load_data/tfrecord) around 100 MB.
    
2. What's the `npz` filepath? ----> `npz_path`
3. What's the key to access the 2-D matrix pointwise data? ----> `npz_key`
4. Where do you want to put the generated tfrecord file? ----> `tfr_path`
5. What's the prefix you want to put in the filename for the tfrecord file? 
    - they will be something like `prefix_0.npz`, ..., `prefix_10.npz`, ...

###  If you have answers to the above in your mind, now you can just call 
### `fh.create_from_npz(num_pts_per_file, npz_path, npz_key, write_tfr_path, prefix)`

In [ ]:
fh.create_from_npz(num_pts_per_file=1e4, npz_path='Big_npz_file.npz',
                   npz_key='data', tfr_path='TFR_dir',
                   prefix="case1")

### Now, you can see the **tfrecord** files that you generated! 

In [ ]:
from pathlib import Path
for path in sorted(Path("TFR_dir").glob("*.tfrecord"), key=lambda path: path.stat().st_mtime):
    print(path.name, path.stat().st_size, "bytes")

# 2. How to train a Keras Model on the tfrecord files that generated above?

### Let's create a toy Model first

In [ ]:
import tensorflow as tf 

x = tf.keras.Input(4,)
l1 = tf.keras.layers.Dense(3, activation='tanh')
model = tf.keras.Model([x], [l1(x)])
model.compile(optimizer='adam', loss='mse')

Next, we generate a **meta dataset** or maybe you can call it **hyper dataset**. We will do something called **sub-dataset-batching** (I called it). 

- First you need to determine how many epoch you want to run ----> `epoch`
- Then we call the method `.get_tfr_meta_dataset(path, epoch)`

In [ ]:
epoch = 2
meta_dataset = fh.get_tfr_meta_dataset(tfr_path='TFR_dir', epoch=epoch)

### Finally, we do sub-dataset batching with `meta_dataset`

- we obtain `batch_file`, which are Tensors basically, from meta_dataset
- we create a `sub` dataset from `batch_file`

- you can add your favourite callbacks if you want
- the default `epoch` in `model.fit` is 1, which makes sense in this context

In [ ]:
callbacks = []
batch_size = 128

for batch_file in meta_dataset:
    batch_dataset = fh.gen_dataset_from_batch_file(batch_file, batch_size)
    model.fit(batch_dataset, verbose='auto', callbacks=callbacks)

### Cautious! 
- it is not perfect since we are `.fit` for each `tfrecord` and this will make epoch information lost after each recall of `.fit`
- we can hack it by using a `callbacks` that explicitly write loss to the disk.. in an append way.
- **again what is presented here is not perfect, I am open for your solutions!**

### Question:

- **Why are you choosing to chop tfrecord like this?** It seems to me it is not what the official document has suggested, the syntax here looks very not software-optimized. 

### **Answer:** 

- Because the standard way of using `tfrecord` will take each data points, i.e., $(t,x,y,z,u,v,w)$ as an `tf.train.Example`. When you have 10 of millions, hundreds of millsion data points, **generating such dataset (not using this dataset for training)** has been reported to have performance issues if your algorithm contains any native Python Loop, which is not avoidable at the current stage. 

- In `nif.data.tfr_dataset.TFRDataset`, we subsampling npz file in a sequential order and take data along each dimension (e.g., (t_0,...,t_M) ) as an `tf.train.Example`, instead of treating each single point `(t,x,y,z,u,v,w)` as an example. So we have 7 `Example` in this demo. 

- In this way, performance issue is greatly reduced. I believe the reason is that we leave the inner looping over all datapoints to the low-level

- So far, I can generate tfrecord from tens of Gigabyte data within 30 mins to 1 hour. While if I tried to make each training example as data point, it may take 

### Let me know <shawnpan@uw.edu>  if you have better solution for this, i.e., without using the meta-dataset to do sub-dataset-batching